In [ ]:
import numpy as np
import pandas as pd

S_BalAcc = pd.read_excel("Tables_forFriedmanNemenyi.xlsx", sheet_name="S_BalAcc", index_col=0)
S_Suc = pd.read_excel("Tables_forFriedmanNemenyi.xlsx", sheet_name="S_Suc", index_col=0)
S_BalAccSD= pd.read_excel("Tables_forFriedmanNemenyi.xlsx", sheet_name="S_BalAccSD", index_col=0)
S_SucSD = pd.read_excel("Tables_forFriedmanNemenyi.xlsx", sheet_name="S_SucSucSD", index_col=0)
R_BalAcc = pd.read_excel("Tables_forFriedmanNemenyi.xlsx", sheet_name="R_BalAcc", index_col=0)

In [ ]:
import scipy.stats as ss
import scikit_posthocs as sp
import matplotlib.pyplot as plt

from matplotlib import rc
plt.rcParams['font.serif'] = ['CMU Serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
plt.rcParams['font.size'] = 11

In [ ]:
def evaluate_fs_methods(df, benchmark_name, save_file, ascending=False):
    """
    Executes the Friedman test, Nemenyi post-hoc, and plots the CD diagram.
    Expects a DataFrame where Rows = Datasets and Columns = FS Methods.

    Imagine the case where the pulse rate of a group of seven students was measured:
     - before exercise (BE), 
     - immediately after exercise (IAE) and 
     - 5 minutes after exercise (AE). 
     
    Is there evidence to suggest that the pulse rates on these three occasions are 
    similar?

    In that case we feed the `ss.friedmanchisquare` function:
    [<7 measurements BE>, <7 measurements IAE>, <7 measurements AE>]

    In our case, the different "occasions" are the FS methods. The seven different 
    "students" are the various datasets.

    Interpreting the Critical Difference diagram:
    The diagram arranges the average ranks of multiple groups on the x axis in order to 
    facilitate performance comparisons between them. The groups that could not be 
    statistically deemed as different are linked by a horizontal crossbar.
    """
    print(f"\n=== Statistical Evaluation: {benchmark_name} ===")
    
    # 1. Friedman Test
    # Each column pertains to each respective feature selection method
    stat, p_value = ss.friedmanchisquare(*[df[col] for col in df.columns])
    print(f"Friedman Test p-value: {p_value}")

    if p_value < 0.05:
        print("Significant difference detected (p < 0.05). Generating Nemenyi CD Diagram...")
        
        # 2. Calculate Average Ranks
        # - ascending = False for Bal.Acc
        # - ascending = True for Standard Deviations, because lower means FS is more robuts
        avg_ranks = df.rank(axis=1, ascending=ascending).mean()

        # 3. Nemenyi Post-Hoc Test Matrix
        # Creates a pairwise matrix of p-values comparing every FS method
        nemenyi_matrix = sp.posthoc_nemenyi_friedman(df)
        
        # 4. Critical Difference (CD) Diagram
        # https://scikit-posthocs.readthedocs.io/en/latest/generated/scikit_posthocs.critical_difference_diagram.html
        plt.figure(figsize=(10, 3))
        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=nemenyi_matrix,
            label_fmt_left='{label} [{rank:.2f}]',
            label_fmt_right='{label} [{rank:.2f}]',
            text_h_margin=0.3,
            crossbar_props={'color': None},
            marker_props={'s': 25, 'color': 'gray', 'edgecolor': 'k'},
        )

        # Make ProD bold
        ax = plt.gca() # Get the current axes generated by the plot
        
        for text in ax.texts:
            label = text.get_text()

            setbold = False
            if 'ProD-Sco' in label:
                setbold = True
            elif 'ProD-Slv' in label:
                setbold = True

            # Change text to black
            text.set_color('black')
            if setbold:
                text.set_fontweight('bold')
            
            # Get the exact X, Y coordinates of the text
            tx, ty = text.get_position()
            
            # Find the line (stem) that connects to this text
            closest_line = None
            min_dist = float('inf')
            
            for line in ax.lines:
                xdata = line.get_xdata()
                ydata = line.get_ydata()
                
                # Calculate distance from the text to the line's endpoints
                for lx, ly in zip(xdata, ydata):
                    dist = (lx - tx)**2 + (ly - ty)**2
                    if dist < min_dist:
                        min_dist = dist
                        closest_line = line
            
            # Color the closest connecting line
            if closest_line is not None:
                closest_line.set_color('black')

                if setbold:
                    closest_line.set_linewidth(2.0) # Slightly thicker, ProD stands out
                else:
                    closest_line.set_linewidth(0.75)
                
                # closest_line.set_zorder(5) # Brings the colored line to the front
        
        plt.title(f'{benchmark_name}', fontsize="large", loc="left")
        plt.tight_layout()
        plt.savefig(f"{save_file}.pdf", format="pdf")
        plt.show()
    else:
        print("No statistically significant differences detected among the methods (p >= 0.05).")

In [ ]:
evaluate_fs_methods(S_BalAcc, "Synthetic Experiments (Bal. Accuracy)", "SBalAcc")

In [ ]:
evaluate_fs_methods(S_Suc, r"Synthetic Experiments ($Suc.$)", "SSuc")

In [ ]:
evaluate_fs_methods(S_BalAccSD, "Synthetic Experiments (Bal. Accuracy SD)", "SBalAccSD", ascending=True)

In [ ]:
evaluate_fs_methods(S_SucSD, r"Synthetic Experiments ($Suc.$ SD)", "SSucSD", ascending=True)

In [ ]:
evaluate_fs_methods(R_BalAcc, r"Real Experiments (Bal. Accuracy)", "RBalAcc")